PLEASE REFER TO APPENDIX ON HOW TO RUN THE CODE, CHANGE THE FILE NAMES ACCORDINGLY IN ORDER FOR THE DATASETS TO BE IMPORTED ON YOUR SYSTEM. YOU CAN LOCATE THE WHEREABOUTS OF THE CSV'S BY CHECKING THE PATHWAY WHERE THE CODE IS TRYING TO IMPORT


USE localized_url function if there's an error with the imports

In [ ]:
import os
def localized_url(url):

    current_dir = os.getcwd().split("statistical-analysis-and-modeling-of-football-data")[0] + "statistical-analysis-and-modeling-of-football-data"
    branch = url.split("statistical-analysis-and-modeling-of-football-data")[1]
    return current_dir + branch

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ScraperFC as sfc

In [3]:
understat=sfc.Understat()
epl_2324=understat.scrape_matches(league='EPL', year='2023/2024')

2023/2024 EPL matches: 100%|██████████| 380/380 [04:08<00:00,  1.53it/s]


In [4]:
test_links=list(epl_2324.keys())

In [5]:
epl_2324[test_links[0]].keys()

dict_keys(['shots_data', 'match_info', 'rosters_data'])

In [9]:
match_info=pd.DataFrame([epl_2324[test_links[i]]['match_info'] for i in range(len(test_links))])

In [10]:
match_info.to_csv('understat_match_2324.csv', index=False)

In [11]:
match_info.columns

Index(['id', 'fid', 'h', 'a', 'date', 'league_id', 'season', 'h_goals',
       'a_goals', 'team_h', 'team_a', 'h_xg', 'a_xg', 'h_w', 'h_d', 'h_l',
       'league', 'h_shot', 'a_shot', 'h_shotOnTarget', 'a_shotOnTarget',
       'h_deep', 'a_deep', 'a_ppda', 'h_ppda'],
      dtype='object')

In [ ]:
test_roaster_df_H=pd.DataFrame(epl_2324[test_links[0]]['rosters_data']['h']).T
test_roaster_df_A=pd.DataFrame(epl_2324[test_links[0]]['rosters_data']['a']).T
test_roaster_df=pd.concat([test_roaster_df_H,test_roaster_df_A])
test_roaster_df['link']=test_links[0]
test_roaster_df

,id,goals,own_goals,shots,xG,time,player_id,team_id,position,player,...,red_card,roster_in,roster_out,key_passes,assists,xA,xGChain,xGBuildup,positionOrder,link
603603,603603,0,0,0,0,90,9077,92,GK,James Trafford,...,0,0,0,0,0,0,0.0639837458729744,0.0639837458729744,1,https://understat.com/match/22275
603604,603604,0,0,0,0,90,5568,92,DR,Connor Roberts,...,0,0,0,0,0,0,0.12203856557607651,0.12203856557607651,2,https://understat.com/match/22275
603606,603606,0,0,0,0,79,6557,92,DC,Louis Beyer,...,0,603614,0,0,0,0,0,0,3,https://understat.com/match/22275
603607,603607,0,0,0,0,90,8756,92,DC,Dara O&#039;Shea,...,0,0,0,0,0,0,0.12203856557607651,0.12203856557607651,3,https://understat.com/match/22275
603605,603605,0,0,0,0,90,11699,92,DC,Ameen Al Dakhil,...,0,0,0,0,0,0,0.12203856557607651,0.12203856557607651,3,https://understat.com/match/22275
603608,603608,0,0,0,0,90,11700,92,DL,Vitinho,...,0,0,0,1,0,0.05805481970310211,0.11931239068508148,0.06125757470726967,4,https://understat.com/match/22275
603609,603609,0,0,1,0.05805481970310211,66,10620,92,MR,Luca Koleosho,...,0,603616,0,2,0,0.15887397527694702,0.21692879498004913,0,8,https://understat.com/match/22275
603610,603610,0,0,0,0,90,1018,92,MC,Josh Cullen,...,0,0,0,0,0,0,0.0639837458729744,0.0639837458729744,9,https://understat.com/match/22275
603611,603611,0,0,0,0,90,8285,92,MC,Sander Berge,...,0,603618,0,0,0,0,0.05805481970310211,0.05805481970310211,9,https://understat.com/match/22275
603612,603612,0,0,2,0.1147281676530838,90,7498,92,ML,Lyle Foster,...,0,603617,0,1,0,0.0639837458729744,0.2367667406797409,0.12203856557607651,10,https://understat.com/match/22275


In [50]:
from tqdm import tqdm
def scrape_to_df(start_season, end_season, league):
    """
    Fetch match metadata and player-level rosters from Understat across a range of seasons.

    Parameters:
        start_season (str): e.g., "2015/2016"
        end_season (str): e.g., "2023/2024"
        league (str): 'EPL', 'La_liga', 'Bundesliga', etc.

    Returns:
        match_info_df (DataFrame): Match-level data (xG, team, date, etc.)
        rosters_data_df (DataFrame): Player-level data per match
    """

    start = int(start_season.split('/')[0])
    end = int(end_season.split('/')[0]) + 1

    match_info_list = []
    rosters_data_list = []

    for year in range(start, end):
        season = f"{year}/{year + 1}"
        print(f"📦 Scraping {season} - {league}")
        
        scraped_dict = understat.scrape_matches(league=league, year=season)
        links = list(scraped_dict.keys())

        # Match info for each match in the season
        match_info_season = [scraped_dict[link]['match_info'] for link in links]
        match_info_list.append(pd.DataFrame(match_info_season))

        # Player-level data
        for link in tqdm(links, desc=f"🔍 Roster data {season}"):
            try:
                home_players = pd.DataFrame(scraped_dict[link]['rosters_data']['h']).T
                away_players = pd.DataFrame(scraped_dict[link]['rosters_data']['a']).T
                combined = pd.concat([home_players, away_players], ignore_index=True)
                combined['match_link'] = link
                rosters_data_list.append(combined)
            except Exception as e:
                print(f"⚠️ Error parsing roster for {link}: {e}")

    # Combine all seasons
    match_info_df = pd.concat(match_info_list, ignore_index=True)
    rosters_data_df = pd.concat(rosters_data_list, ignore_index=True)

    return match_info_df, rosters_data_df
        
    

In [51]:
match_info_df, rosters_data_df = scrape_to_df('2015/2016', '2023/2024', 'EPL')

📦 Scraping 2015/2016 - EPL


🔍 Roster data 2015/2016: 100%|██████████| 380/380 [00:00<00:00, 1509.37it/s]


📦 Scraping 2016/2017 - EPL


🔍 Roster data 2016/2017: 100%|██████████| 380/380 [00:00<00:00, 1617.14it/s]


📦 Scraping 2017/2018 - EPL


...

🔍 Roster data 2021/2022: 100%|██████████| 380/380 [00:00<00:00, 1295.98it/s]


📦 Scraping 2022/2023 - EPL


🔍 Roster data 2022/2023: 100%|██████████| 380/380 [00:00<00:00, 1137.21it/s]


📦 Scraping 2023/2024 - EPL


🔍 Roster data 2023/2024: 100%|██████████| 380/380 [00:00<00:00, 1184.99it/s]


In [12]:
def nan_analysis(df):
    """
    This function takes a DataFrame and returns a DataFrame with the percentage of NaN values for each column.
    Only columns with NaN values are included in the output DataFrame.
    """
    nan_percentage = df.isna().mean() * 100
    nan_percentage = nan_percentage[nan_percentage > 0].reset_index()
    nan_percentage.columns = ['Column Name', 'Percentage of NaN Values']
    nan_percentage = nan_percentage.sort_values(by='Percentage of NaN Values', ascending=False)
    return nan_percentage

In [55]:
match_info_df.to_csv('understat_match_1524.csv', index=False)
rosters_data_df.to_csv('understat_roster_1524.csv', index=False)

In [13]:
match_info_df=pd.read_csv('understat_match_1524.csv')
rosters_data_df=pd.read_csv('understat_roster_1524.csv')

In [15]:
def compare_columns(df1, df2):
    """
    Compare columns of two DataFrames and return the columns that are different.
    """
    df1_columns = set(df1.columns)
    df2_columns = set(df2.columns)

    only_in_df1 = df1_columns - df2_columns
    only_in_df2 = df2_columns - df1_columns

    print(f"Columns only in df1: {only_in_df1}")
    print(f"Columns only in df2: {only_in_df2}")

    return only_in_df1, only_in_df2

In [16]:
compare_columns(match_info_df, match_info)

Columns only in df1: set()
Columns only in df2: set()


(set(), set())

In [4]:
match_info_df=pd.read_csv('understat_match_1524.csv')

In [20]:
columns_list=list(match_info_df.columns)
match_info_df[columns_list[0:15]].head(2)

,id,fid,h,a,date,league_id,season,h_goals,a_goals,team_h,team_a,h_xg,a_xg,h_w,h_d
0,81,958431,89,82,2015-08-08 15:45:00,1,2015,1,0,Manchester United,Tottenham,0.627539,0.674600,0.2859,0.3905
1,82,958427,73,71,2015-08-08 18:00:00,1,2015,0,1,Bournemouth,Aston Villa,0.876106,0.782253,0.3435,0.3607


In [21]:
match_info_df[columns_list[15:]]

,h_l,league,h_shot,a_shot,h_shotOnTarget,a_shotOnTarget,h_deep,a_deep,a_ppda,h_ppda
0,0.3236,EPL,9,9,1,4,4,10,8.2188,13.8261
1,0.2958,EPL,11,7,2,3,11,2,11.8462,6.9000
2,0.2675,EPL,10,11,5,5,5,4,17.1579,6.6500
3,0.1521,EPL,19,11,8,5,5,6,9.5556,10.8800
4,0.6380,EPL,17,11,6,7,5,10,10.6250,5.7368
...,...,...,...,...,...,...,...,...,...,...
3415,0.0579,EPL,15,8,9,2,11,0,30.3000,14.4783
3416,0.0004,EPL,36,4,14,3,22,1,16.6000,5.0968
3417,0.1521,EPL,15,15,6,7,4,5,10.5000,13.8947
3418,0.0185,EPL,28,3,12,2,16,0,63.0000,8.0000
